<a href="https://colab.research.google.com/github/ekerzendorfer/MOL_LAB/blob/main/MOL_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title ▶️ Installation (robust)
import sys, subprocess, os, stat, shutil

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Widgets & Viewer
pip_install(["py3Dmol", "ipywidgets"])

# RDKit: erst pip ("rdkit"), sonst micromamba/conda-forge
try:
    import rdkit
    print("✅ RDKit vorhanden:", rdkit.__version__)
except Exception:
    print("ℹ️ RDKit fehlt → Versuch: pip install rdkit ...")
    try:
        pip_install(["rdkit"])
        import rdkit
        print("✅ RDKit via pip:", rdkit.__version__)
    except Exception:
        print("⚠️ pip fehlgeschlagen → Fallback: micromamba + conda-forge ...")
        # micromamba holen
        os.makedirs("/content/bin", exist_ok=True)
        subprocess.check_call("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content", shell=True)

        # Kandidaten suchen (tar legt manchmal /content/bin/bin/micromamba an)
        candidates = ["/content/bin/micromamba", "/content/bin/bin/micromamba", "/content/micromamba"]
        src = next((c for c in candidates if os.path.isfile(c)), None)
        if not src:
            raise RuntimeError("micromamba konnte nicht installiert werden (curl/tar).")

        dst = "/content/bin/micromamba"
        if src != dst:
            shutil.move(src, dst)

        os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

        prefix = "/content/rdkit_prefix"
        subprocess.check_call([dst, "create", "-y", "-p", prefix, "-c", "conda-forge", "python=3.10", "rdkit"],
                              stdout=subprocess.DEVNULL)

        site = f"{prefix}/lib/python3.10/site-packages"
        if site not in sys.path:
            sys.path.insert(0, site)

        import rdkit
        print("✅ RDKit via conda-forge:", rdkit.__version__)


✅ RDKit vorhanden: 2025.09.3


In [4]:
# @title ▶️ Start MOL_LAB (GitHub → fallback lokal)
import urllib.request, urllib.error, os

USER = "ekerzendorfer"
REPO = "MOL_LAB"
PYFILE = "molekuellabor.py"

def try_download(branch):
    url = f"https://raw.githubusercontent.com/{USER}/{REPO}/{branch}/{PYFILE}"
    try:
        urllib.request.urlretrieve(url, PYFILE)
        print(f"✅ Geladen aus GitHub: {url}")
        return True
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print(f"⚠️ Nicht gefunden (404): {url}")
            return False
        raise

ok = try_download("main") or try_download("master")

if not ok:
    if os.path.exists(PYFILE):
        print("✅ Verwende lokale Datei molekuellabor.py (z.B. hochgeladen oder im Dateibaum vorhanden).")
    else:
        raise FileNotFoundError(
            "molekuellabor.py wurde weder in GitHub gefunden (main/master) noch lokal gefunden.\n"
            "➡️ Lösung: molekuellabor.py ins Repo committen ODER in Colab hochladen."
        )

exec(open(PYFILE, "r", encoding="utf-8").read())



✅ Geladen aus GitHub: https://raw.githubusercontent.com/ekerzendorfer/MOL_LAB/main/molekuellabor.py
